In [21]:
import ccxt
import pandas as pd
import numpy as np
import ta
import time

In [22]:
exchange = ccxt.binance({
    'options': {
        'defaultType': 'future'
    },
    'enableRateLimit': True
})

In [23]:
markets = exchange.load_markets()

symbols = []

for symbol in markets:

    market = markets[symbol]

    if (
        market['quote'] == 'USDT'
        and market['active']
        and market['contract']
    ):
        symbols.append(symbol)

print(f"Total USDT Futures Pairs: {len(symbols)}")

Total USDT Futures Pairs: 583


In [24]:
def fetch_ohlcv(symbol, timeframe='15m', limit=200):

    try:
        ohlcv = exchange.fetch_ohlcv(
            symbol,
            timeframe=timeframe,
            limit=limit
        )

        df = pd.DataFrame(
            ohlcv,
            columns=[
                'timestamp',
                'open',
                'high',
                'low',
                'close',
                'volume'
            ]
        )

        df['timestamp'] = pd.to_datetime(
            df['timestamp'],
            unit='ms'
        )

        return df

    except Exception as e:

        print(f"Error fetching {symbol}: {e}")

        return None

In [25]:
def add_moving_averages(df):

    df['ma7'] = df['close'].rolling(7).mean()

    df['ma25'] = df['close'].rolling(25).mean()

    df['ma99'] = df['close'].rolling(99).mean()

    return df

In [26]:
def add_atr(df, atr_length=14):

    atr_indicator = ta.volatility.AverageTrueRange(
        high=df['high'],
        low=df['low'],
        close=df['close'],
        window=atr_length
    )

    df['atr'] = atr_indicator.average_true_range()

    df['atr_average'] = df['atr'].rolling(14).mean()

    return df

In [27]:
def add_volatility(df, volatility_length=20):

    highest_price = df['high'].rolling(volatility_length).max()

    lowest_price = df['low'].rolling(volatility_length).min()

    df['volatility_percent'] = (
        (highest_price - lowest_price)
        / df['close']
    ) * 100

    return df

In [28]:
def detect_trend(
    df,
    atr_multiplier=1.2,
    min_volatility=0.8
):

    latest = df.iloc[-1]

    # MA STRUCTURE
    bull_trend = (
        latest['ma7'] > latest['ma25']
        and latest['ma25'] > latest['ma99']
        and latest['close'] > latest['ma7']
    )

    bear_trend = (
        latest['ma7'] < latest['ma25']
        and latest['ma25'] < latest['ma99']
        and latest['close'] < latest['ma7']
    )

    # ATR EXPANSION
    high_atr = (
        latest['atr']
        > latest['atr_average'] * atr_multiplier
    )

    # VOLATILITY
    good_volatility = (
        latest['volatility_percent']
        > min_volatility
    )

    # FINAL CONDITIONS
    strong_bull = (
        bull_trend
        and high_atr
        and good_volatility
    )

    strong_bear = (
        bear_trend
        and high_atr
        and good_volatility
    )

    if strong_bull:
        return "BULLISH"

    elif strong_bear:
        return "BEARISH"

    else:
        return None

In [29]:
results = []

for symbol in symbols:

    print(f"Scanning {symbol}")

    df = fetch_ohlcv(symbol)

    if df is None:
        continue

    df = add_moving_averages(df)

    df = add_atr(df)

    df = add_volatility(df)

    trend = detect_trend(df)

    if trend:

        latest = df.iloc[-1]

        results.append({
            'symbol': symbol,
            'trend': trend,
            'price': latest['close'],
            'atr': round(latest['atr'], 4),
            'volatility': round(
                latest['volatility_percent'],
                2
            )
        })

    time.sleep(0.1)

Scanning BTC/USDT:USDT


Scanning ETH/USDT:USDT
Scanning BCH/USDT:USDT
Scanning XRP/USDT:USDT
Scanning LTC/USDT:USDT
Scanning TRX/USDT:USDT
Scanning ETC/USDT:USDT
Scanning LINK/USDT:USDT
Scanning XLM/USDT:USDT
Scanning ADA/USDT:USDT
Scanning XMR/USDT:USDT
Scanning DASH/USDT:USDT
Scanning ZEC/USDT:USDT
Scanning XTZ/USDT:USDT
Scanning BNB/USDT:USDT
Scanning ATOM/USDT:USDT
Scanning ONT/USDT:USDT
Scanning IOTA/USDT:USDT
Scanning BAT/USDT:USDT
Scanning VET/USDT:USDT
Scanning NEO/USDT:USDT
Scanning QTUM/USDT:USDT
Scanning IOST/USDT:USDT
Scanning THETA/USDT:USDT
Scanning ALGO/USDT:USDT
Scanning ZIL/USDT:USDT
Scanning KNC/USDT:USDT
Scanning ZRX/USDT:USDT
Scanning COMP/USDT:USDT
Scanning DOGE/USDT:USDT
Scanning KAVA/USDT:USDT
Scanning BAND/USDT:USDT
Scanning RLC/USDT:USDT
Scanning SNX/USDT:USDT
Scanning DOT/USDT:USDT
Scanning YFI/USDT:USDT
Scanning CRV/USDT:USDT
Scanning TRB/USDT:USDT
Scanning RUNE/USDT:USDT
Scanning SUSHI/USDT:USDT
Scanning EGLD/USDT:USDT
Scanning SOL/USDT:USDT
Scanning ICX/USDT:USDT
Scanning STORJ/US

In [30]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='volatility',
    ascending=False
)

results_df

,symbol,trend,price,atr,volatility
1,COMP/USDT:USDT,BEARISH,19.60,0.2410,11.79
0,BCH/USDT:USDT,BEARISH,365.16,1.8291,3.59
2,DIS/USDT:USDT,BEARISH,102.40,0.3009,1.15
